# Instacart market-basket TPP split artifacts

이 노트북은 `sample_data/insta_market_basket/instacart_marked_target_df.parquet`를 quantity reconstruction 실험용 `train / validation / test` parquet로 분리합니다.

Instacart 데이터는 user별 basket-count event sequence입니다. 기존 `mark`가 있더라도 본 실험에서는 `demand_qty`에서 magnitude mark와 residual을 다시 계산하여 세 quantity 데이터셋의 target 정의를 통일합니다.

## 1. Setup

노트북 실행 위치와 관계없이 project root와 preprocessing utility를 import합니다.

In [1]:
from pathlib import Path
import sys

def resolve_project_root(start: Path | None = None) -> Path:
    """Find the project root even when Jupyter starts from home or /tmp."""
    anchor = (start or Path.cwd()).expanduser().resolve()
    base = anchor if anchor.is_dir() else anchor.parent
    candidates = [base, *base.parents]
    candidates.extend([
        Path('~/workspace/paper_research').expanduser(),
        Path('/home/workspace/paper_research'),
        Path('/Users/igwanhyeong/PycharmProjects/paper_research'),
    ])
    for candidate in candidates:
        if all((candidate / name).exists() for name in ('sample_data', 'models', 'utils')):
            return candidate
    raise RuntimeError('Could not locate the paper_research project root.')

PROJECT_ROOT = resolve_project_root()
PREPROCESSING_DIR = PROJECT_ROOT / 'simple_lab_test' / 'notebooks' / 'preprocessing'
for path in (PROJECT_ROOT, PREPROCESSING_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from tpp_split_utils import SplitConfig, build_and_save_quantity_splits, ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)

PROJECT_ROOT = /home/leekwanhyeong/workspace/paper_research


## 2. Split configuration

Instacart는 user 수가 많고 quantity tail은 taxi보다 약하지만, 작은 수량 차이를 더 잘 나누기 위해 기존 실험 정책과 동일하게 `scale_base=2.0`을 사용합니다.

In [2]:
cfg = SplitConfig(
    dataset_name='insta_market_basket',
    input_path=PROJECT_ROOT / 'sample_data' / 'insta_market_basket' / 'instacart_marked_target_df.parquet',
    output_dir=PROJECT_ROOT / 'sample_data' / 'insta_market_basket',
    output_prefix='instacart_marked_target',
    scale_base=2.0,
    train_ratio=0.70,
    validation_ratio=0.15,
    test_ratio=0.15,
)
cfg

SplitConfig(dataset_name='insta_market_basket', input_path=PosixPath('/home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_df.parquet'), output_dir=PosixPath('/home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket'), output_prefix='instacart_marked_target', scale_base=2.0, train_ratio=0.7, validation_ratio=0.15, test_ratio=0.15, entity_col='oper_part_no', order_col='seq', clip_min_qty=1.0, min_count=100, min_coverage=0.999)

## 3. Build and save artifacts

이 셀은 split column이 있는 전체 파일과 split별 parquet 파일을 저장합니다. 기존 `instacart_marked_target_with_split.parquet`가 있으면 최신 magnitude rule 기준으로 다시 씁니다.

In [3]:
result = build_and_save_quantity_splits(cfg, overwrite=True)
print('max_order fitted on train =', result['max_order'])
for name, path in result['paths'].items():
    print(f'{name}: {path}')

max_order fitted on train = 7
with_split: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_with_split.parquet
train: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_train.parquet
validation: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_validation.parquet
test: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_test.parquet
manifest: /home/leekwanhyeong/workspace/paper_research/sample_data/insta_market_basket/instacart_marked_target_split_manifest.json


## 4. Quality checks

split별 row 수와 mark 분포를 확인합니다. Instacart는 짧은 user sequence가 많기 때문에 일부 user는 validation/test에 충분히 기여하지 못할 수 있습니다.

In [4]:
from IPython.display import display
import polars as pl

labeled_df = result['labeled_df']
display(labeled_df.head())
display(pl.DataFrame(result['summary']['split_counts']))
display(pl.DataFrame(result['summary']['mark_counts']).pivot(values='len', index='mark', columns='chronological_split').fill_null(0).sort('mark'))
display(labeled_df.select([
    pl.len().alias('rows'),
    pl.col('oper_part_no').n_unique().alias('series'),
    pl.col('demand_qty').median().alias('qty_median'),
    pl.col('demand_qty').quantile(0.95).alias('qty_p95'),
    pl.col('demand_qty').max().alias('qty_max'),
    pl.col('scale_residual').min().alias('residual_min'),
    pl.col('scale_residual').max().alias('residual_max'),
]))

oper_part_no,demand_dt,seq,delta_t,demand_qty,chronological_split,log_qty,log10_qty,raw_order,mark,scale_residual,z
str,i64,i64,i32,f64,str,f64,f64,i32,i32,f64,f64
"""user_1""",0,0,0,5.0,"""train""",2.321928,0.69897,2,2,0.321928,2.321928
"""user_1""",15,15,15,6.0,"""train""",2.584963,0.778151,2,2,0.584963,2.584963
"""user_1""",36,36,21,5.0,"""train""",2.321928,0.69897,2,2,0.321928,2.321928
"""user_1""",65,65,29,5.0,"""train""",2.321928,0.69897,2,2,0.321928,2.321928
"""user_1""",93,93,28,8.0,"""train""",3.0,0.90309,3,3,0.0,3.0


chronological_split,rows,series,qty_median,qty_p95,qty_max
str,i64,i64,f64,f64,f64
"""test""",578387,196615,9.0,26.0,177.0
"""train""",2197401,206209,8.0,25.0,175.0
"""validation""",503733,206195,9.0,26.0,155.0


/tmp/ipykernel_1260390/3830830770.py:7: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  display(pl.DataFrame(result['summary']['mark_counts']).pivot(values='len', index='mark', columns='chronological_split').fill_null(0).sort('mark'))


mark,test,train,validation
i64,i64,i64,i64
0,23558,105240,22012
1,61996,268057,57468
2,152664,613146,136557
3,210517,787329,181601
4,115692,384748,95300
5,13695,38198,10567
6,259,674,225
7,6,9,3


rows,series,qty_median,qty_p95,qty_max,residual_min,residual_max
u32,u32,f64,f64,f64,f64,f64
3279521,206209,8.0,25.0,177.0,0.0,0.988685
